# 09. SacreBLEU, chrF++, & COMET Benchmark Evaluation

**Requires GPU; requires gated-repo access only if evaluating Llama.** This notebook downloads an 8B-parameter model. **Qwen/Qwen2.5-7B-Instruct** is fully open (no authentication needed); **meta-llama/Llama-3.1-8B-Instruct** is gated on HuggingFace Hub -- before evaluating Llama:
1. Visit https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while logged in and request/accept access.
2. Authenticate this environment: run `huggingface-cli login` in a terminal/shell cell, or set the `HF_TOKEN` environment variable to a token from https://huggingface.co/settings/tokens.

Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

Evaluates one model/checkpoint against the fixed `master_test.csv` split and saves results for the ablation study (notebook 11).

In [ ]:
# ============================================================
# PATH BOOSTER -- guarantees project root in sys.path & CWD.
# Matches notebooks/00_setup_environment.ipynb's convention: this repo
# is deployed both to Colab (fresh `git clone`, folder "Ekegusii-LLM-Translation")
# and to Kineses Cloud / similar Jupyter hosts (pre-placed at
# ~/Ekegusii-LLM-Translation-main -- the "-main" suffix comes from GitHub's
# "Download ZIP" naming). Do not assume either folder name is the cwd.
# ============================================================
import os
import sys

REPO_NAME = "Ekegusii-LLM-Translation"


def _find_project_root():
    if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
        if not os.path.exists(REPO_NAME):
            os.system(f"git clone https://github.com/aykahsay/{REPO_NAME}.git")
            os.system(f"pip install -q -r {REPO_NAME}/requirements.txt")
        return os.path.abspath(REPO_NAME)

    try:
        cwd = os.getcwd()
    except FileNotFoundError:
        cwd = os.path.expanduser("~")
        os.chdir(cwd)

    home = os.path.expanduser("~")
    for candidate in (f"{REPO_NAME}-main", REPO_NAME):
        proj_dir = os.path.join(home, candidate)
        if os.path.isdir(proj_dir):
            return proj_dir

    if os.path.exists("src") and os.path.exists("data"):
        return cwd
    if os.path.basename(cwd) == "notebooks" and os.path.exists(os.path.join("..", "src")):
        return os.path.abspath("..")

    return cwd


project_root = _find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Project root: {project_root}")


In [ ]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


In [ ]:
MODEL_NAME = 'qwen'              # 'qwen' or 'llama'
EXPERIMENT_ID = 'E4_Trilingual'  # must match a completed training run, or None for zero-shot (E0)
ADAPTER_PATH = f'checkpoints/{MODEL_NAME}/{EXPERIMENT_ID}/best' if EXPERIMENT_ID else None

In [ ]:
from src.cli.evaluate import run_evaluate

results = run_evaluate(MODEL_NAME, source_lang='English', target_lang='Ekegusii', adapter_path=ADAPTER_PATH)
results

## Save results for the ablation study

In [ ]:
import json
from pathlib import Path

out_dir = Path('experiments') / (EXPERIMENT_ID or 'E0_Baseline')
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'results.json'

existing = json.loads(results_path.read_text()) if results_path.exists() else {'experiment_id': EXPERIMENT_ID}
existing[MODEL_NAME] = results
results_path.write_text(json.dumps(existing, indent=2))
print(f'Saved to {results_path}')

## COMET (optional, slower -- downloads a ~1.7GB checkpoint on first use)

In [ ]:
from src.experiments.base import BaseExperiment
from src.master_corpus.manager import MasterCorpusManager
from src.evaluation.comet import CometEvaluator
from src.models.qwen.inference import translate_with_qwen
from src.models.llama.inference import translate_with_llama

class _EvalHelper(BaseExperiment):
    experiment_id = 'notebook09'
    def build_training_tasks(self): raise NotImplementedError

helper = _EvalHelper(MasterCorpusManager())
test_pairs = helper.build_test_pairs('English', 'Ekegusii')
sources, references = test_pairs['source'].tolist(), test_pairs['target'].tolist()

translate_fn = translate_with_qwen if MODEL_NAME == 'qwen' else translate_with_llama
predictions = translate_fn(sources, 'English', 'Ekegusii', adapter_path=ADAPTER_PATH)

comet_result = CometEvaluator().compute(predictions, references, sources)
print(f"Mean COMET: {comet_result['mean_score']:.4f}")